### Question 3.1 — CVAE Architecture for Tiny-ImageNet-10 (64 x 64 x 3)

This solution is split across multiple cells (not a single monolithic cell).

Design constraints satisfied:
- Encoder uses `Conv2d(..., stride=2, ...)` for downsampling.
- Decoder uses `ConvTranspose2d(..., stride=2, ...)` for upsampling.
- Final decoder activation is `Sigmoid` (pixel range `[0, 1]`).

In [1]:
import torch
import torch.nn as nn

# You can tune this value as needed.
LATENT_DIM = 128

In [2]:
class Encoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1),   # 64 -> 32
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),  # 32 -> 16
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # 16 -> 8
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),# 8 -> 4
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),# 4 -> 2
            nn.ReLU(inplace=True),
            nn.Conv2d(512, latent_dim, kernel_size=2, stride=1, padding=0), # 2 -> 1
        )

    def forward(self, x):
        return self.features(x)  # [B, latent_dim, 1, 1]

In [3]:
class Decoder(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.recon = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, kernel_size=2, stride=1, padding=0), # 1 -> 2
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1), # 2 -> 4
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1), # 4 -> 8
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),  # 8 -> 16
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),   # 16 -> 32
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),    # 32 -> 64
            nn.Sigmoid(),
        )

    def forward(self, z):
        return self.recon(z)  # [B, 3, 64, 64]

In [4]:
class CVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

In [11]:
def summarize_sequential_shapes(seq_model, input_shape):
    """Return a layer-by-layer shape trace for an nn.Sequential module."""
    x = torch.zeros(input_shape)
    rows = []
    for idx, layer in enumerate(seq_model):
        in_shape = tuple(x.shape)
        x = layer(x)
        out_shape = tuple(x.shape)
        rows.append((idx, layer.__class__.__name__, in_shape, out_shape))
    return rows


def print_cvae_summary(latent_dim=LATENT_DIM, batch_size=1):
    model = CVAE(latent_dim=latent_dim)

    enc_rows = summarize_sequential_shapes(model.encoder.features, (batch_size, 3, 64, 64))
    dec_rows = summarize_sequential_shapes(model.decoder.recon, (batch_size, latent_dim, 1, 1))

    print("=" * 90)
    print("CVAE ARCHITECTURE SUMMARY")
    print("=" * 90)
    print(f"Input image      : ({batch_size}, 3, 64, 64)")
    print(f"Latent tensor    : ({batch_size}, {latent_dim}, 1, 1)")
    print(f"Reconstruction   : ({batch_size}, 3, 64, 64)")

    print("\n[Encoder Trace]")
    for idx, name, in_s, out_s in enc_rows:
        print(f"{idx:02d} | {name:<18} | {in_s} -> {out_s}")

    print("\n[Decoder Trace]")
    for idx, name, in_s, out_s in dec_rows:
        print(f"{idx:02d} | {name:<18} | {in_s} -> {out_s}")


print_cvae_summary()

CVAE ARCHITECTURE SUMMARY
Input image      : (1, 3, 64, 64)
Latent tensor    : (1, 128, 1, 1)
Reconstruction   : (1, 3, 64, 64)

[Encoder Trace]
00 | Conv2d             | (1, 3, 64, 64) -> (1, 32, 32, 32)
01 | ReLU               | (1, 32, 32, 32) -> (1, 32, 32, 32)
02 | Conv2d             | (1, 32, 32, 32) -> (1, 64, 16, 16)
03 | ReLU               | (1, 64, 16, 16) -> (1, 64, 16, 16)
04 | Conv2d             | (1, 64, 16, 16) -> (1, 128, 8, 8)
05 | ReLU               | (1, 128, 8, 8) -> (1, 128, 8, 8)
06 | Conv2d             | (1, 128, 8, 8) -> (1, 256, 4, 4)
07 | ReLU               | (1, 256, 4, 4) -> (1, 256, 4, 4)
08 | Conv2d             | (1, 256, 4, 4) -> (1, 512, 2, 2)
09 | ReLU               | (1, 512, 2, 2) -> (1, 512, 2, 2)
10 | Conv2d             | (1, 512, 2, 2) -> (1, 128, 1, 1)

[Decoder Trace]
00 | ConvTranspose2d    | (1, 128, 1, 1) -> (1, 512, 2, 2)
01 | ReLU               | (1, 512, 2, 2) -> (1, 512, 2, 2)
02 | ConvTranspose2d    | (1, 512, 2, 2) -> (1, 256, 4, 4)
03 |

### Quick RGB (3-Channel) Verification

This check proves the model is configured for RGB data:
- Encoder first conv expects `in_channels = 3`
- Decoder last transposed conv produces `out_channels = 3`
- A dummy RGB tensor of shape `(B, 3, 64, 64)` passes through the model successfully

In [10]:
def verify_rgb_handling(latent_dim=LATENT_DIM, batch_size=2):
    model = CVAE(latent_dim=latent_dim)

    # Structural checks
    enc_first = model.encoder.features[0]      # Conv2d
    dec_last = model.decoder.recon[-2]         # Final ConvTranspose2d (before Sigmoid)

    print(f"Encoder first layer in_channels  : {enc_first.in_channels}")
    print(f"Decoder final layer out_channels : {dec_last.out_channels}")

    # End-to-end shape check with RGB input
    x_rgb = torch.randn(batch_size, 3, 64, 64)
    x_hat, z = model(x_rgb)

    print(f"Input shape    : {tuple(x_rgb.shape)}")
    print(f"Latent shape   : {tuple(z.shape)}")
    print(f"Output shape   : {tuple(x_hat.shape)}")

    assert enc_first.in_channels == 3, "Encoder is not configured for RGB input."
    assert dec_last.out_channels == 3, "Decoder is not producing RGB output."
    assert x_hat.shape[1] == 3, "Output does not have 3 channels."

    print("\nRGB check passed: model handles 3-channel 64x64 images.")


verify_rgb_handling()

Encoder first layer in_channels  : 3
Decoder final layer out_channels : 3
Input shape    : (2, 3, 64, 64)
Latent shape   : (2, 128, 1, 1)
Output shape   : (2, 3, 64, 64)

RGB check passed: model handles 3-channel 64x64 images.
